# Embeddings in LLMs

Embeddings convert tokens (words/subwords) into vectors (lists of numbers).
These vectors capture **meaning & relationships** so the model can do math on them.

Examples of what embeddings can capture:
- "king" is close to "queen"
- "dog" is close to "puppy"
- "car" is far from "banana"

LLMs cannot work directly with text — they only understand **numbers → vectors → matrices**.
So embeddings are the **bridge between text and math**.

_Text → Tokenization → Embeddings → Transformer (attention & layers) → Logits → Decoding → Detokenization → Output text_


## Part 1 — A Toy Embedding

We start with a very simple example where we manually assign vectors to words.
This helps build intuition for what embeddings *do*.

In [ ]:
import numpy as np

# Tiny vocabulary
vocab = ["cat", "dog", "banana", "apple"]

# Fake embeddings: each word -> 2-D vector
# In real models, these numbers are learned. Here I just choose simple values.
embeddings = {
    "cat": np.array([1.0, 1.0]),
    "dog": np.array([1.2, 0.9]),
    "banana": np.array([-1.0, 0.5]),
    "apple": np.array([-1.2, 0.4]),
}

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors.
    Cosine similarity ~ 1.0 means very similar, ~ 0 means unrelated, ~ -1 means opposite.
    """
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similarity(cat, dog) =", cosine_similarity(embeddings["cat"], embeddings["dog"]))
print("Similarity(cat, banana) =", cosine_similarity(embeddings["cat"], embeddings["banana"]))

### Takeaway
- Words become **vectors**.
- Similar words have **similar vectors**.
- We can measure similarity using **cosine similarity**.

Of course, these are toy examples; real embeddings are high-dimensional (e.g., 256, 768, 1024+ dimensions).

## Part 2 — What Does Cosine Similarity Mean?

Cosine similarity measures the **angle** between two vectors, not their length.
This is useful because we care more about the *direction* (meaning) than magnitude (intensity/scale).

In [ ]:
import numpy as np

a = np.array([1, 0])
b = np.array([0.9, 0.1])
c = np.array([-1, 0])

def cos_sim(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print("similar (a, b):", cos_sim(a, b))
print("opposite (a, c):", cos_sim(a, c))

### Takeaway
- **1.0** → vectors point in the same direction (very similar meaning)
- **0.0** → vectors are orthogonal (unrelated)
- **-1.0** → vectors point in opposite directions (opposite meaning)

## Part 3 — Learnable Embeddings (Like LLMs Do)

In real neural networks, embeddings are **learned automatically during training**.
An embedding layer is essentially a **lookup table of vectors** that gets updated
with gradient descent.

A tiny demo with PyTorch.

In [ ]:
import torch

# Suppose vocab has 5 tokens, embedding size = 3 - this will create a learning embedding matrix of shape (5, 3)
embedding_layer = torch.nn.Embedding(num_embeddings=5, embedding_dim=3)

# Token IDs (e.g., "cat"=0, "dog"=1, etc.)
tokens = torch.tensor([0, 1, 2])

# Look up their embeddings
embeds = embedding_layer(tokens)

print("Embedding matrix (all token vectors):\n", embedding_layer.weight)
print("\nToken embeddings for [0, 1, 2]:\n", embeds)

### Takeaway
> An embedding layer is just a **learnable lookup table**.

- Each row in `embedding_layer.weight` is the vector for one token.
- During training, these vectors are updated so that similar tokens end up with similar vectors.


## Part 4 — Real Token Embeddings from a Transformer Model

Next, let us look at embeddings produced by a real Transformer model.
We'll use a small BERT-like model (DistilBERT) from Hugging Face.

In [ ]:
# If running for the first time, you can uncomment and run this cell to install transformers
# !pip install transformers

from transformers import AutoTokenizer, AutoModel
import torch

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "Large language models are powerful. Large large very large."

# Tokenize the input text
inputs = tokenizer(text, return_tensors="pt") # pt = PyTorch tensors

# Get model outputs (includes hidden states)
with torch.no_grad():
    outputs = model(**inputs)

last_hidden_state = outputs.last_hidden_state  # (batch_size, seq_len, hidden_dim)

print("Token IDs:", inputs["input_ids"])
print("Embedding shape:", last_hidden_state.shape)

### Takeaway
- Shape is `(batch_size, sequence_length, embedding_dim)`.
- Each token (after tokenization) gets a vector of size 768 for DistilBERT.
- These are **contextual embeddings**: the same word can have different vectors in different sentences.
- Some embedding models are **uncased**, so they convert everything to lower case before tokenization. However, most SOTA models are cased. 

## Part 5 — Same tokens, different embeddings

We send two different sentences into DistilBERT:

- one where “Apple” = fruit
- one where “Apple” = company

DistilBERT converts each token into a context-dependent embedding using self-attention.
We then extract the embedding for the word “apple” in each sentence and measure cosine similarity between them.

In [ ]:
# pip install transformers torch

from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
model = AutoModel.from_pretrained("distilbert-base-uncased")

fruit = "Apple is good for health."
company = "Apple released a new iPhone yesterday."

# Tokenize
enc_fruit = tokenizer(fruit, return_tensors="pt")
enc_company = tokenizer(company, return_tensors="pt")

# ===== 1: INPUT (STATIC) EMBEDDINGS =====
embedding_layer = model.get_input_embeddings()   # shape: (vocab, hidden_dim)

# Lookup embedding vectors before the transformer
inp_embed_fruit = embedding_layer(enc_fruit["input_ids"])
inp_embed_company = embedding_layer(enc_company["input_ids"])

# ===== 2: CONTEXTUAL EMBEDDINGS =====
with torch.no_grad():
    ctx_embed_fruit = model(**enc_fruit).last_hidden_state
    ctx_embed_company = model(**enc_company).last_hidden_state

# ===== 3: FIND TOKEN INDEX FOR "apple" =====
tokens_fruit = tokenizer.convert_ids_to_tokens(enc_fruit["input_ids"][0])
tokens_company = tokenizer.convert_ids_to_tokens(enc_company["input_ids"][0])

idx_fruit = tokens_fruit.index("apple")
idx_company = tokens_company.index("apple")

# ===== 4: GET VECTORS =====
# Input (static)
vec_input_fruit = inp_embed_fruit[0, idx_fruit, :]
vec_input_company = inp_embed_company[0, idx_company, :]

# Contextual (final layer)
vec_ctx_fruit = ctx_embed_fruit[0, idx_fruit, :]
vec_ctx_company = ctx_embed_company[0, idx_company, :]

# ===== 5: COSINE SIMILARITY =====
sim_input = F.cosine_similarity(vec_input_fruit, vec_input_company, dim=0)
sim_ctx = F.cosine_similarity(vec_ctx_fruit, vec_ctx_company, dim=0)

print("Static input embedding similarity:", sim_input.item())
print("Contextual embedding similarity:", sim_ctx.item())
print("Fruit tokens:", tokens_fruit)
print("Company tokens:", tokens_company)

## Part 6 — Sentence Embeddings (Meaning of Whole Sentences)

Sometimes we want **one vector per sentence**, not per token.
We can use dedicated models like `sentence-transformers` for this.

In [ ]:
# If running for the first time, uncomment and run this cell to install sentence-transformers
# !pip install sentence-transformers

from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

sent1 = "The cat is sleeping."
sent2 = "A dog is napping."
sent3 = "I love pizza."

emb1 = model.encode(sent1)
emb2 = model.encode(sent2)
emb3 = model.encode(sent3)

def cos(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Similarity (cat vs dog sentences):", cos(emb1, emb2))
print("Similarity (cat vs pizza sentences):", cos(emb1, emb3))

### Takeaway
- Similar sentences → higher cosine similarity.
- Unrelated sentences → lower similarity.
- Sentence embeddings are useful for search, clustering, recommendation, etc.

## Part 7 — Visualizing Embeddings

To build intuition, we can visualize high-dimensional embeddings in 2D using t-SNE.
This is just for understanding; in real systems we don't usually do this.

In [ ]:
# !pip install sentence-transformers matplotlib scikit-learn

from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

words = ["king", "queen", "man", "woman", "apple", "banana", "orange"]
vecs = model.encode(words)

tsne = TSNE(n_components=2, perplexity=5, random_state=0)
points = tsne.fit_transform(vecs)

plt.figure(figsize=(6, 6))
plt.scatter(points[:, 0], points[:, 1])

for word, (x, y) in zip(words, points):
    plt.annotate(word, (x, y))

plt.title("2D visualization of word embeddings (t-SNE)")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.show()

### Takeaway
- Words with related meanings tend to cluster together in this 2D projection.
- This is a **visual intuition** for how embeddings organize concepts in space.

## Summary

- **Embeddings convert text into dense vectors of numbers.**
- Similar meanings → similar vectors.
- An embedding layer in an LLM is a **learned lookup table**.
- Transformers use these embeddings as the input to attention and other components.
- We can compute similarity, search, cluster, or visualize using these vectors.